# 05 Final Prediction

This notebook selects the best ranked candidate, rolls the clean prefix forward deterministically, extracts the 192 answer bits, and saves both the final answer and best candidate for submission packaging.

In [2]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

SEARCH_ROOTS = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(path for path in SEARCH_ROOTS if (path / "DAY2_data.txt").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sparsetap_config import DEFAULT_CONFIG
from sparsetap_utils import (
    extract_answer_bits,
    load_candidates,
    prepare_environment,
    rank_candidates,
    rollout_from_prefix,
    save_final_answer,
)

config = DEFAULT_CONFIG
data, prefix = prepare_environment(config)
ranked = load_candidates(config.candidate_dir / "all_ranked_candidates.json")
best = rank_candidates(ranked, data=data, prefix=prefix, noise=config.noise_prob)[0]

rolled = rollout_from_prefix(prefix, best["taps"], total_len=256)
answer = extract_answer_bits(rolled, start=64, end=256)

save_final_answer(
    answer,
    best,
    config.final_dir / "final_answer.txt",
    config.final_dir / "best_candidate.json",
)

print("Best tap set:", best["taps"])
print("Final 192-bit prediction string:")
print(answer)
best


Best tap set: [63]
Final 192-bit prediction string:
000010100011010010101100101001110100011110010110011010000111010000010100011010010101100101001110100011110010110011010000111010000010100011010010101100101001110100011110010110011010000111010000


{'candidate_id': 'predictive-single_lag_scan-e1e66a9f',
 'track': 'predictive',
 'method': 'single_lag_scan',
 'taps': [63],
 'W': 63,
 'scores': {'prefix_consistent': 1,
  'log_likelihood': -353288.96972742194,
  'accuracy': 0.5007461139896373,
  'num_taps': 1,
  'rank_tuple': (1, -353288.96972742194, 0.5007461139896373, -1)},
 'metadata': {'note': 'Simple one-lag agreement baseline.',
  'lag': 63,
  'source_scores': {'single_lag_match_rate': 0.5007461139896373,
   'single_lag_lift': 0.0007461139896373092,
   'prefix_consistency': 1,
   'log_likelihood': -353288.96972742194,
   'accuracy': 0.5007461139896373,
   'num_taps': 1,
   'rank_tuple': [1, -353288.96972742194, 0.5007461139896373, -1]}}}